# SpendDNA – Your Wallet's Year-End Story

## Minor Project

**Student Name:** Akanksha Manjunath Nayak  
**Project:** SpendDNA  
**Dataset Persona:** Rahul Sharma  
**Platform:** Google Colab  
**Technology:** Python, Pandas and NumPy  
**Dataset Period:** January 2024 – June 2024

# Project Requirement Checklist

### Mandatory Features

- [ ] Transaction Parser
- [ ] Date Cleaning
- [ ] Amount Cleaning
- [ ] Type Standardisation
- [ ] Duplicate Removal
- [ ] Vendor Extractor
- [ ] P2P Transfer Detection
- [ ] ATM Withdrawal Detection
- [ ] Category Tagger
- [ ] Spending Overview
- [ ] Monthly Trend Analysis
- [ ] Time-of-Day Analysis
- [ ] NumPy Activity Heatmap
- [ ] Anomaly Detection
- [ ] Spending Archetype Detection

### Bonus Features

- [ ] Day-of-Week Analysis
- [ ] Vendor Cleanup Audit
- [ ] Custom Bengaluru Archetype
- [ ] NumPy Spending Forecast

### Final Submission

- [ ] Final Console Report
- [ ] Three Data-Specific Insights
- [ ] Reflection
- [ ] AI Disclosure
- [ ] Final Validation

In [1]:
# AI-assisted: Basic notebook setup and library selection.

import pandas as pd
import numpy as np
from datetime import datetime

print("Libraries imported successfully.")

Libraries imported successfully.


## Dataset Loading

The original transaction file is loaded into a Pandas DataFrame.

In [3]:
file_path = "rahul_transactions.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

Dataset loaded successfully.
Number of rows: 1328
Number of columns: 8


## Initial Dataset Inspection

Before cleaning the data, we inspect the structure, columns and transaction fields.

In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nTransaction types:")
print(df["Type"].value_counts())

print("\nUnique modes:")
print(df["Mode"].value_counts(dropna=False))

Column names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

First 5 rows:


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962



Data types:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object

Transaction types:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64

Unique modes:
Mode
UPI     1299
ATM       17
NEFT       6
IMPS       6
Name: count, dtype: int64


# Feature 1 – Transaction Parser

### Step 1: Date Cleaning

The dataset contains multiple date formats. We convert them into one standard datetime format.

In [5]:
# Convert mixed date formats into a standard datetime column.
# format="mixed" allows Pandas to parse different formats in the same column.

df["date_clean"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

print("Unparseable dates:", df["date_clean"].isna().sum())
print(df[["Date", "date_clean"]].head(10))

Unparseable dates: 0
          Date date_clean
0   2024-01-01 2024-01-01
1    01-Jan-24 2024-01-01
2    01-Jan-24 2024-01-01
3   2024-01-01 2024-01-01
4  01 Jan 2024 2024-01-01
5   2024-01-01 2024-01-01
6   2024-01-01 2024-01-01
7    01-Jan-24 2024-01-01
8   2024-01-02 2024-01-02
9     02/01/24 2024-01-02


### Step 2: Amount Cleaning

The Amount column contains rupee symbols, Rs. prefixes, commas and plain numbers.

In [6]:
df["amount_clean"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount_clean"] = pd.to_numeric(
    df["amount_clean"],
    errors="coerce"
)

print("Unparseable amounts:", df["amount_clean"].isna().sum())

print(df[["Amount", "amount_clean"]].head(10))

Unparseable amounts: 0
    Amount  amount_clean
0    ₹2462        2462.0
1    50.00          50.0
2   ₹84728       84728.0
3    ₹1828        1828.0
4   270.00         270.0
5  Rs. 625         625.0
6  Rs. 148         148.0
7     ₹482         482.0
8  Rs. 537         537.0
9     3956        3956.0


### Step 3: Transaction Type Standardisation

The dataset contains DR, Debit and CR variants.

In [7]:
def standardize_type(value):
    value = str(value).strip().lower()

    if value in ["dr", "debit"]:
        return "debit"

    elif value in ["cr", "credit"]:
        return "credit"

    else:
        return "unknown"


df["type_clean"] = df["Type"].apply(standardize_type)

print(df["type_clean"].value_counts())

type_clean
debit     1322
credit       6
Name: count, dtype: int64


### Step 4: Duplicate Removal

The project dataset contains exact duplicate transactions. These are removed before analysis.

In [8]:
rows_before = len(df)

duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().copy()

rows_after = len(df)

print("Rows before removing duplicates:", rows_before)
print("Duplicate rows found:", duplicate_count)
print("Rows after removing duplicates:", rows_after)

Rows before removing duplicates: 1328
Duplicate rows found: 18
Rows after removing duplicates: 1310


In [9]:
# Remove rows where essential values could not be parsed.
df = df.dropna(subset=["date_clean", "amount_clean"]).copy()

# Create useful date/time features.
df["month_number"] = df["date_clean"].dt.month
df["month_name"] = df["date_clean"].dt.strftime("%b")
df["day_of_week"] = df["date_clean"].dt.day_name()

# The Time column is already HH:MM.
df["hour"] = pd.to_numeric(
    df["Time"].astype(str).str[:2],
    errors="coerce"
)

print(df[[
    "date_clean",
    "month_number",
    "month_name",
    "day_of_week",
    "hour"
]].head())

  date_clean  month_number month_name day_of_week  hour
0 2024-01-01             1        Jan      Monday     3
1 2024-01-01             1        Jan      Monday     5
2 2024-01-01             1        Jan      Monday     9
3 2024-01-01             1        Jan      Monday    14
4 2024-01-01             1        Jan      Monday    14


In [10]:
print("=" * 60)
print("FEATURE 1 VALIDATION")
print("=" * 60)

print("Clean transactions:", len(df))
print("Unparseable dates:", df["date_clean"].isna().sum())
print("Unparseable amounts:", df["amount_clean"].isna().sum())
print("Unknown transaction types:",
      (df["type_clean"] == "unknown").sum())
print("Duplicates remaining:", df.duplicated().sum())
print("Months found:", sorted(df["month_number"].unique()))

FEATURE 1 VALIDATION
Clean transactions: 1310
Unparseable dates: 0
Unparseable amounts: 0
Unknown transaction types: 0
Duplicates remaining: 0
Months found: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


# Feature 2 – Vendor Extractor

The Description field contains messy merchant names.
We convert those descriptions into canonical vendor names using
dictionaries, loops and conditional logic.

In [11]:
unique_descriptions = sorted(df["Description"].unique())

print("Number of unique descriptions:",
      len(unique_descriptions))

print("\nSample descriptions:")
for description in unique_descriptions[:100]:
    print(description)

Number of unique descriptions: 283

Sample descriptions:
AIRTEL POSTPAID
AMAZON IN
AMAZON PRIME VIDEO
AMAZON SELLER SVCS
AMAZONIN MARKETPLACE
AMZN PRIME
AMZN-INTPYMT
ANI Technologies
ATM-WDL-HDFC-3609
ATM-WDL-HDFC-4942
ATM-WDL-HDFC-8030
ATM-WDL-HDFC-8253
ATM-WDL-HDFC-9140
ATM-WDL-ICICI-3918
ATM-WDL-ICICI-4172
ATM-WDL-ICICI-4739
ATM-WDL-ICICI-5025
ATM-WDL-ICICI-6478
ATM-WDL-ICICI-9135
ATM-WDL-SBI-0237
ATM-WDL-SBI-0279
ATM-WDL-SBI-0874
ATM-WDL-SBI-4080
ATM-WDL-SBI-4084
ATM-WDL-SBI-5715
AVENUE SUPERMARTS
Amazon Pay India
BANGALORE ELEC SUPPLY
BESCOM ELEC BILL
BHARTI AIRTEL LTD
BHIM SWIGGY
BHIM ZEPTO
BHIM-BLINKIT
BHIM-BMTC
BIGBASKET BANGALORE
BIGTREE ENTERTAINMENT
BLINKIT BANGALORE
BMS MOVIE TICKETS
BMTC BUS PASS
BUNDL TECH-INSTAMART
BUNDL Tech P L
BWSSB WATER BILL
COFFEE DAY GLOBAL
DISNEY HOTSTAR
FKART INTRNET
FLIPKART INDIA
FSN E-COMMERCE
Flipkart Internet
GROFERS INDIA P L
GROWW INVEST TECH
IMPS ZERODHA-COIN
IMPS-RENT-LANDLORD-35126704
IMPS-RENT-LANDLORD-36852906
IMPS-RENT-LANDLORD-3959

In [12]:
vendor_patterns = {

    "Swiggy": [
        "SWIGGY",
        "BUNDL TECH P L"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    "Instamart": [
        "INSTAMART"
    ],

    "Zepto": [
        "ZEPTO",
        "KIRANAKART"
    ],

    "Blinkit": [
        "BLINKIT",
        "GROFERS"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN"
    ],

    "Flipkart": [
        "FLIPKART",
        "FKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Nykaa": [
        "NYKAA",
        "FSN E-COMMERCE"
    ],

    "BigBasket": [
        "BIGBASKET"
    ],

    "DMart": [
        "DMART",
        "AVENUE SUPERMARTS",
        "INNOVATIVE RETAIL"
    ],

    "Ola": [
        "OLA",
        "ANI TECHNOLOGIES"
    ],

    "Uber": [
        "UBER"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "BMTC": [
        "BMTC",
        "TUMMOC"
    ],

    "Coffee Day": [
        "COFFEE DAY",
        "CCD"
    ],

    "Starbucks": [
        "STARBUCKS"
    ],

    "Third Wave Coffee": [
        "THIRDWAVE",
        "THIRD WAVE",
        "TWC INDIA"
    ],

    "Restaurant": [
        "RESTAURANT",
        "DINEOUT",
        "EMPIRE",
        "MEGHANA",
        "TRUFFLES"
    ],

    "Netflix": [
        "NETFLIX"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "Disney Hotstar": [
        "HOTSTAR",
        "DISNEY HOTSTAR"
    ],

    "Star India": [
        "STAR INDIA"
    ],

    "Jio": [
        "JIOFIBER",
        "JIORECHARGE",
        "RELIANCE JIO"
    ],

    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Vodafone Idea": [
        "VODAFONE IDEA",
        "VI POSTPAID",
        "UPI-VI-RECHARGE"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC"
    ],

    "BWSSB": [
        "BWSSB"
    ],

    "Zerodha": [
        "ZERODHA"
    ],

    "Groww": [
        "GROWW"
    ],

    "BPCL": [
        "BPCL"
    ],

    "HP": [
        "HP PETROL"
    ],

    "Indian Oil": [
        "INDIAN OIL",
        "IOC"
    ],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS MOVIE"
    ],

    "BigTree Entertainment": [
        "BIGTREE ENTERTAINMENT"
    ]
}

print("Vendor patterns created:", len(vendor_patterns))

Vendor patterns created: 35


In [13]:
p2p_keywords = [
    "AMAN-",
    "ANKIT-",
    "KARAN-",
    "NEHA-",
    "PRIYA-",
    "SNEHA-",
    "VIKAS-"
]


def extract_vendor(description):

    text = str(description).upper().strip()

    # ATM withdrawals
    if text.startswith("ATM-WDL-"):
        return "Cash Withdrawal"

    # Salary transactions
    if "SALARY" in text:
        return "Salary Credit"

    # Rent transactions
    if "RENT-LANDLORD" in text:
        return "Rent"

    # Person-to-person transfers
    if text.startswith("UPI-"):
        for person in p2p_keywords:
            if person in text:
                return "P2P Transfer"

    # Merchant matching
    for vendor, keywords in vendor_patterns.items():

        for keyword in keywords:

            if keyword in text:
                return vendor

    return "Uncategorised"

In [14]:
df["vendor_clean"] = df["Description"].apply(
    extract_vendor
)

print("Unique canonical vendors:",
      df["vendor_clean"].nunique())

print("\nTop vendors:")
print(
    df["vendor_clean"]
    .value_counts()
    .head(10)
)

Unique canonical vendors: 40

Top vendors:
vendor_clean
Swiggy        214
Zomato        121
Ola            87
Amazon         86
Restaurant     73
Zepto          71
Uber           71
Blinkit        55
Flipkart       47
Starbucks      42
Name: count, dtype: int64


In [15]:
uncategorised = df[
    df["vendor_clean"] == "Uncategorised"
]

print("Uncategorised transactions:",
      len(uncategorised))

print("\nUncategorised descriptions:")

for description in sorted(
    uncategorised["Description"].unique()
):
    print(description)

Uncategorised transactions: 14

Uncategorised descriptions:
ROPPEN TRANSPORTATION


# Feature 3 – Category Tagger

Each canonical vendor is assigned to a spending category.

In [16]:
category_map = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",

    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Ola": "Transport",
    "Uber": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    "Coffee Day": "Cafe",
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",

    "Restaurant": "Restaurants",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney Hotstar": "Subscriptions",
    "Star India": "Subscriptions",

    "Jio": "Utilities",
    "Airtel": "Utilities",
    "Vodafone Idea": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Rent": "Utilities",

    "BigBasket": "Groceries",
    "DMart": "Groceries",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "BPCL": "Fuel",
    "HP": "Fuel",
    "Indian Oil": "Fuel",

    "BookMyShow": "Entertainment",
    "BigTree Entertainment": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    "Salary Credit": "Income"
}

In [17]:
df["category"] = (
    df["vendor_clean"]
    .map(category_map)
    .fillna("Uncategorised")
)

print("Category counts:")
print(df["category"].value_counts())

Category counts:
category
Food Delivery        335
Transport            236
E-commerce           172
Quick Commerce       155
Cafe                  99
Restaurants           73
Utilities             49
Groceries             41
Subscriptions         31
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Uncategorised         14
Entertainment         13
Income                 6
Name: count, dtype: int64


In [18]:
print("=" * 60)
print("CATEGORY VALIDATION")
print("=" * 60)

print(
    "Number of categories:",
    df["category"].nunique()
)

print("\nCategory list:")
for category in sorted(df["category"].unique()):
    print("-", category)

CATEGORY VALIDATION
Number of categories: 16

Category list:
- Cafe
- Cash Withdrawal
- E-commerce
- Entertainment
- Food Delivery
- Fuel
- Groceries
- Income
- Investments
- Personal Transfer
- Quick Commerce
- Restaurants
- Subscriptions
- Transport
- Uncategorised
- Utilities


# Feature 4 – Spending Overview

We calculate the major financial statistics:

- Total credits
- Total debits
- Net change
- Savings rate
- Top categories
- Top vendors
- Transaction count

In [19]:
credit_df = df[df["type_clean"] == "credit"].copy()
debit_df = df[df["type_clean"] == "debit"].copy()

total_credits = credit_df["amount_clean"].sum()
total_debits = debit_df["amount_clean"].sum()

net_change = total_credits - total_debits

if total_credits != 0:
    savings_rate = (net_change / total_credits) * 100
else:
    savings_rate = 0

print("Total credits :", f"₹{total_credits:,.2f}")
print("Total debits  :", f"₹{total_debits:,.2f}")
print("Net change    :", f"₹{net_change:,.2f}")
print("Savings rate  :", f"{savings_rate:.2f}%")
print("Transactions  :", len(df))
print("Unique vendors:", df["vendor_clean"].nunique())

Total credits : ₹509,774.00
Total debits  : ₹1,678,901.00
Net change    : ₹-1,169,127.00
Savings rate  : -229.34%
Transactions  : 1310
Unique vendors: 40


In [20]:
consumption_df = debit_df[
    ~debit_df["category"].isin(
        ["Personal Transfer", "Cash Withdrawal"]
    )
].copy()

category_spend = (
    consumption_df
    .groupby("category")["amount_clean"]
    .sum()
    .sort_values(ascending=False)
)

print("TOP 5 CATEGORIES")
print("-" * 50)

for category, amount in category_spend.head(5).items():

    percentage = (
        amount /
        consumption_df["amount_clean"].sum()
    ) * 100

    print(
        f"{category:<20} "
        f"{percentage:>6.2f}%   "
        f"₹{amount:>12,.2f}"
    )

TOP 5 CATEGORIES
--------------------------------------------------
E-commerce            37.54%   ₹  603,877.00
Investments           15.43%   ₹  248,160.00
Utilities              9.32%   ₹  149,914.00
Food Delivery          9.09%   ₹  146,249.00
Restaurants            7.32%   ₹  117,737.00


In [21]:
vendor_spend = (
    consumption_df
    .groupby("vendor_clean")["amount_clean"]
    .sum()
    .sort_values(ascending=False)
)

print("TOP 5 VENDORS")
print("-" * 50)

for vendor, amount in vendor_spend.head(5).items():

    order_count = (
        consumption_df[
            consumption_df["vendor_clean"] == vendor
        ].shape[0]
    )

    print(
        f"{vendor:<22} "
        f"₹{amount:>12,.2f} "
        f"({order_count} transactions)"
    )

TOP 5 VENDORS
--------------------------------------------------
Amazon                 ₹  328,530.00 (86 transactions)
Zerodha                ₹  210,000.00 (14 transactions)
Flipkart               ₹  177,510.00 (47 transactions)
Restaurant             ₹  117,737.00 (73 transactions)
Rent                   ₹  108,000.00 (6 transactions)


In [22]:
months = [1, 2, 3, 4, 5, 6]

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

categories = sorted(
    consumption_df["category"].unique()
)

monthly_matrix = np.zeros(
    (len(categories), len(months))
)

for i, category in enumerate(categories):

    for j, month in enumerate(months):

        amount = consumption_df[
            (consumption_df["category"] == category) &
            (consumption_df["month_number"] == month)
        ]["amount_clean"].sum()

        monthly_matrix[i, j] = amount


monthly_df = pd.DataFrame(
    monthly_matrix,
    index=categories,
    columns=[month_names[m] for m in months]
)

display(monthly_df.round(2))

,Jan,Feb,Mar,Apr,May,Jun
Cafe,3690.0,4273.0,5448.0,6564.0,5668.0,5802.0
E-commerce,98623.0,94011.0,108215.0,69219.0,95776.0,138033.0
Entertainment,1263.0,474.0,2418.0,2224.0,0.0,1914.0
Food Delivery,22076.0,23740.0,23553.0,27302.0,24193.0,25385.0
Fuel,30322.0,2079.0,26164.0,18718.0,9138.0,2882.0
Groceries,17649.0,8571.0,6289.0,11748.0,9718.0,5432.0
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0
Quick Commerce,11611.0,15177.0,15276.0,12324.0,13162.0,10922.0
Restaurants,16320.0,21772.0,28313.0,7711.0,22286.0,21335.0
Subscriptions,2767.0,4628.0,3509.0,2168.0,1844.0,3555.0


In [23]:
trend_results = []

for i, category in enumerate(categories):

    first_month = monthly_matrix[i, 0]
    last_month = monthly_matrix[i, -1]

    if first_month != 0:
        growth = (
            (last_month - first_month)
            / first_month
        ) * 100
    else:
        growth = np.nan

    trend_results.append(
        (category, growth)
    )

trend_results = [
    item for item in trend_results
    if not np.isnan(item[1])
]

trend_results.sort(
    key=lambda x: x[1],
    reverse=True
)

print("BIGGEST GROWTH:")
print(trend_results[0])

print("\nBIGGEST DECLINE:")
print(trend_results[-1])

BIGGEST GROWTH:
('Cafe', np.float64(57.23577235772358))

BIGGEST DECLINE:
('Fuel', np.float64(-90.49534991095575))


# Feature 6 – Time-of-Day Spending Patterns

We build a 24-hour category activity matrix using NumPy.

Rows = categories
Columns = hours 00–23

In [24]:
hours = list(range(24))

activity_matrix = np.zeros(
    (len(categories), 24),
    dtype=int
)

for i, category in enumerate(categories):

    for hour in hours:

        count = consumption_df[
            (consumption_df["category"] == category) &
            (consumption_df["hour"] == hour)
        ].shape[0]

        activity_matrix[i, hour] = count


print("Activity matrix shape:")
print(activity_matrix.shape)

Activity matrix shape:
(13, 24)


In [25]:
print("=" * 90)
print("NUMPY ACTIVITY HEATMAP")
print("=" * 90)

for i, category in enumerate(categories):

    row = activity_matrix[i]

    max_value = row.max()

    if max_value == 0:
        bars = [""] * 24
    else:
        bars = [
            "#" * max(1, int((value / max_value) * 10))
            if value > 0 else ""
            for value in row
        ]

    print(f"\n{category}")

    for hour, bar in enumerate(bars):

        print(
            f"{hour:02d}:00 | {bar}"
        )

NUMPY ACTIVITY HEATMAP

Cafe
00:00 | ##
01:00 | ##
02:00 | 
03:00 | #
04:00 | 
05:00 | 
06:00 | #
07:00 | #
08:00 | ######
09:00 | #####
10:00 | ##########
11:00 | ####
12:00 | ###
13:00 | ###
14:00 | ###
15:00 | #####
16:00 | ######
17:00 | #######
18:00 | #####
19:00 | #
20:00 | ###
21:00 | 
22:00 | 
23:00 | #

E-commerce
00:00 | ########
01:00 | ####
02:00 | ###
03:00 | ########
04:00 | ########
05:00 | #########
06:00 | ######
07:00 | ###
08:00 | ###
09:00 | #########
10:00 | #####
11:00 | ##########
12:00 | ######
13:00 | ####
14:00 | ########
15:00 | ####
16:00 | #######
17:00 | ######
18:00 | ########
19:00 | ###
20:00 | ######
21:00 | ######
22:00 | #####
23:00 | #########

Entertainment
00:00 | #####
01:00 | #####
02:00 | 
03:00 | 
04:00 | 
05:00 | 
06:00 | 
07:00 | #####
08:00 | ##########
09:00 | 
10:00 | 
11:00 | 
12:00 | 
13:00 | 
14:00 | 
15:00 | 
16:00 | ##########
17:00 | 
18:00 | ##########
19:00 | #####
20:00 | ##########
21:00 | 
22:00 | #####
23:00 | 

Food Delivery

In [26]:
food_delivery = consumption_df[
    consumption_df["category"] == "Food Delivery"
].copy()

late_night_food = food_delivery[
    food_delivery["hour"].isin(
        [21, 22, 23, 0, 1, 2]
    )
]

if len(food_delivery) > 0:

    late_night_percentage = (
        len(late_night_food) /
        len(food_delivery)
    ) * 100

else:
    late_night_percentage = 0

print(
    f"Late-night Food Delivery: "
    f"{late_night_percentage:.2f}%"
)

Late-night Food Delivery: 20.90%


In [27]:
consumption_df["category_mean"] = (
    consumption_df
    .groupby("category")["amount_clean"]
    .transform("mean")
)

consumption_df["category_std"] = (
    consumption_df
    .groupby("category")["amount_clean"]
    .transform("std")
)

consumption_df["z_score"] = np.where(
    consumption_df["category_std"] > 0,
    (
        consumption_df["amount_clean"]
        - consumption_df["category_mean"]
    ) / consumption_df["category_std"],
    0
)

print(
    consumption_df[
        [
            "vendor_clean",
            "category",
            "amount_clean",
            "z_score"
        ]
    ].head()
)

  vendor_clean        category  amount_clean   z_score
0       Amazon      E-commerce        2462.0 -0.231951
1         BMTC       Transport          50.0 -1.256888
4      Blinkit  Quick Commerce         270.0 -1.170751
5        Zepto  Quick Commerce         625.0  0.588317
6         Uber       Transport         148.0 -0.597481


In [28]:
anomalies = consumption_df[
    consumption_df["z_score"] > 2
].copy()

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print("=" * 70)
print("TOP ANOMALIES")
print("=" * 70)

for _, row in anomalies.head(5).iterrows():

    date_text = row["date_clean"].strftime(
        "%d %b"
    )

    print(
        f"{date_text:<8} "
        f"{row['vendor_clean']:<20} "
        f"₹{row['amount_clean']:>10,.2f}   "
        f"z = {row['z_score']:.2f}"
    )

print("\nTotal anomalies:", len(anomalies))

TOP ANOMALIES
26 Jun   Amazon               ₹ 22,008.00   z = 4.09
07 Feb   Amazon               ₹ 21,986.00   z = 4.09
26 Feb   Restaurant           ₹  8,383.00   z = 3.88
05 Mar   Amazon               ₹ 19,917.00   z = 3.63
22 Jun   Restaurant           ₹  7,935.00   z = 3.63

Total anomalies: 33


In [29]:
def foodie_archetype(spend, total):
    food_total = (
        spend.get("Food Delivery", 0)
        + spend.get("Restaurants", 0)
        + spend.get("Cafe", 0)
    )

    percentage = (food_total / total) * 100

    return percentage > 25, percentage


def quick_commerce_archetype(spend, total):
    percentage = (
        spend.get("Quick Commerce", 0)
        / total
    ) * 100

    return percentage > 15, percentage


def shopaholic_archetype(spend, total):
    percentage = (
        spend.get("E-commerce", 0)
        / total
    ) * 100

    return percentage > 15, percentage


def investor_archetype(spend, total):
    percentage = (
        spend.get("Investments", 0)
        / total
    ) * 100

    return percentage > 15, percentage


def late_night_snacker(food_data):
    if len(food_data) == 0:
        return False, 0

    late_night = food_data[
        food_data["hour"].isin(
            [21, 22, 23, 0, 1, 2]
        )
    ]

    percentage = (
        len(late_night)
        / len(food_data)
    ) * 100

    return percentage > 50, percentage


def cab_commuter_archetype(spend, total):
    percentage = (
        spend.get("Transport", 0)
        / total
    ) * 100

    return percentage > 10, percentage


def subscription_lover(data):
    vendors = data[
        data["category"] == "Subscriptions"
    ]["vendor_clean"].nunique()

    return vendors >= 5, vendors


def yolo_spender(savings_rate):
    return savings_rate < 10, savings_rate


def disciplined_saver(savings_rate):
    return savings_rate > 40, savings_rate

In [30]:
spend_dict = category_spend.to_dict()

archetypes = []

result, metric = foodie_archetype(
    spend_dict,
    consumption_df["amount_clean"].sum()
)

if result:
    archetypes.append(
        ("THE FOODIE", metric)
    )


result, metric = quick_commerce_archetype(
    spend_dict,
    consumption_df["amount_clean"].sum()
)

if result:
    archetypes.append(
        ("THE QUICK COMMERCE JUNKIE", metric)
    )


result, metric = shopaholic_archetype(
    spend_dict,
    consumption_df["amount_clean"].sum()
)

if result:
    archetypes.append(
        ("THE SHOPAHOLIC", metric)
    )


result, metric = investor_archetype(
    spend_dict,
    consumption_df["amount_clean"].sum()
)

if result:
    archetypes.append(
        ("THE INVESTOR", metric)
    )


result, metric = late_night_snacker(
    food_delivery
)

if result:
    archetypes.append(
        ("THE LATE-NIGHT SNACKER", metric)
    )


result, metric = cab_commuter_archetype(
    spend_dict,
    consumption_df["amount_clean"].sum()
)

if result:
    archetypes.append(
        ("THE CAB COMMUTER", metric)
    )


result, metric = subscription_lover(df)

if result:
    archetypes.append(
        ("THE SUBSCRIPTION LOVER", metric)
    )


result, metric = yolo_spender(savings_rate)

if result:
    archetypes.append(
        ("THE YOLO SPENDER", metric)
    )


result, metric = disciplined_saver(savings_rate)

if result:
    archetypes.append(
        ("THE DISCIPLINED SAVER", metric)
    )


print("=" * 70)
print("SPENDING ARCHETYPES")
print("=" * 70)

for name, metric in archetypes:

    if name == "THE SUBSCRIPTION LOVER":
        print(
            f"→ {name:<30} "
            f"({metric} vendors)"
        )

    elif name == "THE YOLO SPENDER" or \
         name == "THE DISCIPLINED SAVER":

        print(
            f"→ {name:<30} "
            f"({metric:.2f}% savings rate)"
        )

    else:

        print(
            f"→ {name:<30} "
            f"({metric:.2f}%)"
        )

SPENDING ARCHETYPES
→ THE SHOPAHOLIC                 (37.54%)
→ THE INVESTOR                   (15.43%)
→ THE YOLO SPENDER               (-229.34% savings rate)


In [31]:
weekday_spend = (
    consumption_df[
        ~consumption_df["day_of_week"].isin(
            ["Saturday", "Sunday"]
        )
    ]["amount_clean"].sum()
)

weekend_spend = (
    consumption_df[
        consumption_df["day_of_week"].isin(
            ["Saturday", "Sunday"]
        )
    ]["amount_clean"].sum()
)

print("Weekday spending :", f"₹{weekday_spend:,.2f}")
print("Weekend spending :", f"₹{weekend_spend:,.2f}")

if weekday_spend > 0:

    weekend_difference = (
        (weekend_spend - weekday_spend)
        / weekday_spend
    ) * 100

else:
    weekend_difference = 0

print(
    f"Weekend vs weekday difference: "
    f"{weekend_difference:.2f}%"
)

Weekday spending : ₹1,146,975.00
Weekend spending : ₹461,827.00
Weekend vs weekday difference: -59.74%


# Bonus Archetype – The Pavement Coffee Connoisseur

### Rule

A user matches this archetype if they spend at cafes across
4 or more distinct cafe vendors.

In [32]:
cafe_vendors = consumption_df[
    consumption_df["category"] == "Cafe"
]["vendor_clean"].nunique()

custom_archetype = cafe_vendors >= 4

print("Distinct cafe vendors:", cafe_vendors)

if custom_archetype:
    print("→ THE PAVEMENT COFFEE CONNOISSEUR")
else:
    print("Custom archetype not detected.")

Distinct cafe vendors: 3
Custom archetype not detected.


In [33]:
uncategorised_rows = df[
    df["vendor_clean"] == "Uncategorised"
]

print("=" * 60)
print("VENDOR CLEANUP AUDIT")
print("=" * 60)

if len(uncategorised_rows) == 0:

    print("Excellent! No unmapped descriptions found.")

else:

    print(
        "Unmapped descriptions:",
        len(uncategorised_rows)
    )

    for description in sorted(
        uncategorised_rows["Description"].unique()
    ):
        print("→", description)

VENDOR CLEANUP AUDIT
Unmapped descriptions: 14
→ ROPPEN TRANSPORTATION


In [34]:
uncategorised_rows = df[
    df["vendor_clean"] == "Uncategorised"
]

print("=" * 60)
print("VENDOR CLEANUP AUDIT")
print("=" * 60)

if len(uncategorised_rows) == 0:

    print("Excellent! No unmapped descriptions found.")

else:

    print(
        "Unmapped descriptions:",
        len(uncategorised_rows)
    )

    for description in sorted(
        uncategorised_rows["Description"].unique()
    ):
        print("→", description)

VENDOR CLEANUP AUDIT
Unmapped descriptions: 14
→ ROPPEN TRANSPORTATION


# Bonus – 3-Month Spending Forecast

The forecast uses only NumPy arithmetic.

For each category, the next month's spending is estimated
using the average of the latest 3 months.

In [35]:
forecast_results = {}

for i, category in enumerate(categories):

    last_three_months = monthly_matrix[i, -3:]

    forecast = np.mean(last_three_months)

    forecast_results[category] = forecast


print("=" * 70)
print("NEXT MONTH SPENDING FORECAST")
print("=" * 70)

for category, forecast in forecast_results.items():

    print(
        f"{category:<22} "
        f"₹{forecast:>12,.2f}"
    )

NEXT MONTH SPENDING FORECAST
Cafe                   ₹    6,011.33
E-commerce             ₹  101,009.33
Entertainment          ₹    1,379.33
Food Delivery          ₹   25,626.67
Fuel                   ₹   10,246.00
Groceries              ₹    8,966.00
Investments            ₹   42,028.00
Quick Commerce         ₹   12,136.00
Restaurants            ₹   17,110.67
Subscriptions          ₹    2,522.33
Transport              ₹    9,560.00
Uncategorised          ₹      247.00
Utilities              ₹   24,796.67


In [36]:
print("=" * 72)
print("                 SpendDNA REPORT")
print("                    RAHUL SHARMA")
print("=" * 72)

print("\nEXECUTIVE SUMMARY")
print("-" * 72)

print(
    f"Total credits    : ₹{total_credits:,.2f}"
)

print(
    f"Total debits     : ₹{total_debits:,.2f}"
)

print(
    f"Net change       : ₹{net_change:,.2f}"
)

print(
    f"Savings rate     : {savings_rate:.2f}%"
)

print(
    f"Transactions     : {len(df)}"
)

print(
    f"Unique vendors   : {df['vendor_clean'].nunique()}"
)


print("\nTOP CATEGORIES")
print("-" * 72)

consumption_total = consumption_df["amount_clean"].sum()

for category, amount in category_spend.head(5).items():

    percentage = (
        amount / consumption_total
    ) * 100

    bar_length = int(percentage / 2)

    print(
        f"{category:<20} "
        f"{'#' * bar_length:<25} "
        f"{percentage:>6.2f}% "
        f"₹{amount:>10,.2f}"
    )


print("\nTOP VENDORS")
print("-" * 72)

for vendor, amount in vendor_spend.head(5).items():

    print(
        f"{vendor:<22} "
        f"₹{amount:>12,.2f}"
    )


print("\nTIME-OF-DAY PATTERNS")
print("-" * 72)

print(
    f"Food Delivery late-night share : "
    f"{late_night_percentage:.2f}%"
)


print("\nTOP ANOMALIES")
print("-" * 72)

for _, row in anomalies.head(5).iterrows():

    date_text = row["date_clean"].strftime(
        "%d %b"
    )

    print(
        f"{date_text:<8} "
        f"{row['vendor_clean']:<20} "
        f"₹{row['amount_clean']:>10,.2f} "
        f"(z={row['z_score']:.2f})"
    )


print("\nSPENDING ARCHETYPES")
print("-" * 72)

for name, metric in archetypes:

    print(
        f"→ {name}"
    )


print("\nBONUS ARCHETYPE")
print("-" * 72)

if custom_archetype:
    print(
        "→ THE PAVEMENT COFFEE CONNOISSEUR"
    )
else:
    print(
        "Custom archetype not detected."
    )


print("\n" + "=" * 72)
print("                    END OF REPORT")
print("=" * 72)

                 SpendDNA REPORT
                    RAHUL SHARMA

EXECUTIVE SUMMARY
------------------------------------------------------------------------
Total credits    : ₹509,774.00
Total debits     : ₹1,678,901.00
Net change       : ₹-1,169,127.00
Savings rate     : -229.34%
Transactions     : 1310
Unique vendors   : 40

TOP CATEGORIES
------------------------------------------------------------------------
E-commerce           ##################         37.54% ₹603,877.00
Investments          #######                    15.43% ₹248,160.00
Utilities            ####                        9.32% ₹149,914.00
Food Delivery        ####                        9.09% ₹146,249.00
Restaurants          ###                         7.32% ₹117,737.00

TOP VENDORS
------------------------------------------------------------------------
Amazon                 ₹  328,530.00
Zerodha                ₹  210,000.00
Flipkart               ₹  177,510.00
Restaurant             ₹  117,737.00
Rent        

# Key Insights

The following insights are generated from the calculated transaction data.

In [37]:
top_category = category_spend.index[0]
top_category_amount = category_spend.iloc[0]

top_category_percentage = (
    top_category_amount /
    consumption_total
) * 100

top_vendor = vendor_spend.index[0]
top_vendor_amount = vendor_spend.iloc[0]

print("1.",
      f"{top_category} is the highest spending category, "
      f"accounting for {top_category_percentage:.2f}% "
      f"of consumption spending.")

print("2.",
      f"{late_night_percentage:.2f}% of Food Delivery "
      f"transactions occurred during late-night hours.")

print("3.",
      f"{top_vendor} is the highest-spend vendor at "
      f"₹{top_vendor_amount:,.2f}.")

1. E-commerce is the highest spending category, accounting for 37.54% of consumption spending.
2. 20.90% of Food Delivery transactions occurred during late-night hours.
3. Amazon is the highest-spend vendor at ₹328,530.00.


# Reflection

This project helped me understand how messy transaction data can be
cleaned and transformed into meaningful financial insights.

I learned how to parse mixed date and amount formats, standardise
transaction types, normalise merchant names, categorize spending,
analyse monthly and time-based behaviour, detect anomalies using
z-scores, and identify spending archetypes using quantitative rules.

The project also helped me understand the importance of data quality,
business-oriented analysis and clear reporting.

# AI Assistance Disclosure

AI assistance was used for understanding requirements, notebook
organisation, debugging guidance and code-review support.

The analysis was performed on the supplied dataset, and the final
numerical results are generated by executing the notebook code.
Vendor mappings were designed from inspection of the supplied
transaction descriptions.

In [38]:
print("=" * 70)
print("FINAL PROJECT VALIDATION")
print("=" * 70)

checks = {
    "Dataset loaded": len(df) > 0,

    "Duplicates removed":
        df.duplicated().sum() == 0,

    "Dates parsed":
        df["date_clean"].isna().sum() == 0,

    "Amounts parsed":
        df["amount_clean"].isna().sum() == 0,

    "Transaction types valid":
        (df["type_clean"] != "unknown").all(),

    "Vendor column created":
        "vendor_clean" in df.columns,

    "Category column created":
        "category" in df.columns,

    "Monthly matrix created":
        monthly_matrix.shape[1] == 6,

    "24-hour heatmap created":
        activity_matrix.shape[1] == 24,

    "Z-score calculated":
        "z_score" in consumption_df.columns
}

for check_name, result in checks.items():

    status = "PASS" if result else "FAIL"

    print(
        f"{check_name:<35} : {status}"
    )

print("=" * 70)

FINAL PROJECT VALIDATION
Dataset loaded                      : PASS
Duplicates removed                  : PASS
Dates parsed                        : PASS
Amounts parsed                      : PASS
Transaction types valid             : PASS
Vendor column created               : PASS
Category column created             : PASS
Monthly matrix created              : PASS
24-hour heatmap created             : PASS
Z-score calculated                  : PASS
